In [ ]:
# ==============================================================================
# STEP 1: Import Library & Load Dataset
# ==============================================================================
import pandas as pd
import numpy as np

# Membaca dataset
df = pd.read_csv('retail_indonesia_55k.csv')

print("Ukuran data awal:", df.shape)
df.head()

Ukuran data awal: (55000, 37)


,order_id,tanggal_transaksi,tanggal,bulan,bulan_num,tahun,hari,jam,channel_penjualan,store_id,...,total_penjualan,biaya_produksi,biaya_pengiriman,biaya_platform,biaya_packing,total_biaya,profit,margin_persen,metode_pembayaran,event_promo
0,ORD000001,2024-01-01 08:06,2024-01-01,January,1,2024,Senin,8,Tokopedia,ONLINE,...,213600,105000,18000,7476,2000,132476,81124,37.98,Transfer,Year End Sale
1,ORD000002,2024-01-01 08:07,2024-01-01,January,1,2024,Senin,8,Tokopedia,ONLINE,...,453600,225000,22000,15876,5000,267876,185724,40.94,OVO,Year End Sale
2,ORD000003,2024-01-01 08:12,2024-01-01,January,1,2024,Senin,8,Toko,STR011,...,79200,45000,0,0,0,45000,34200,43.18,QRIS,Year End Sale
3,ORD000004,2024-01-01 08:13,2024-01-01,January,1,2024,Senin,8,Toko,STR008,...,96750,50000,0,0,0,50000,46750,48.32,QRIS,Year End Sale
4,ORD000005,2024-01-01 08:14,2024-01-01,January,1,2024,Senin,8,Website,ONLINE,...,359100,180000,30000,8977,4000,222977,136123,37.91,Kartu Kredit,Year End Sale


In [ ]:
# ==============================================================================
# STEP 2: Pembersihan & Konversi Tipe Data (Data Types Fix)
# ==============================================================================

# 1. Konversi kolom tanggal_transaksi ke datetime
df['tanggal_transaksi'] = pd.to_datetime(df['tanggal_transaksi'])
df['tanggal'] = pd.to_datetime(df['tanggal'])

# 2. Hapus spasi berlebih pada data teks (string trimming)
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].astype(str).str.strip()

# 3. Penanganan Missing Values & Duplikasi Data
df.dropna(how='all', inplace=True)
df.drop_duplicates(inplace=True)

print("Jumlah baris setelah penghapusan duplikat:", len(df))

Jumlah baris setelah penghapusan duplikat: 55000


In [ ]:
# ==============================================================================
# STEP 3: Validasi & Koreksi Kalkulasi Finansial
# ==============================================================================

# Rekalkulasi total_penjualan & total_biaya untuk memastikan tidak ada Anomali
df['total_penjualan_calc'] = df['qty'] * df['harga_jual']
df['total_biaya_calc'] = (
    df['biaya_produksi'] +
    df['biaya_pengiriman'] +
    df['biaya_platform'] +
    df['biaya_packing']
)

# Hitung ulang profit dan margin
df['profit_calc'] = df['total_penjualan'] - df['total_biaya']
df['margin_persen_calc'] = np.where(
    df['total_penjualan'] > 0,
    np.round((df['profit'] / df['total_penjualan']) * 100, 2),
    0
)

# Hapus kolom bantuan sementara jika hasil kalkulasi sudah sesuai
df.drop(columns=['total_penjualan_calc', 'total_biaya_calc', 'profit_calc', 'margin_persen_calc'], inplace=True)

In [ ]:
# ==============================================================================
# STEP 4: Cek Ringkasan Data Bersih & Simpan
# ==============================================================================

print("\n--- RINGKASAN DATA BERSIH ---")
print(df.info())

# Simpan hasil data cleaning ke file CSV baru
output_file = 'retail_indonesia_55k_cleaned.csv'
df.to_csv(output_file, index=False)
print(f"\nData berhasil dibersihkan dan disimpan ke '{output_file}'")


--- RINGKASAN DATA BERSIH ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55000 entries, 0 to 54999
Data columns (total 37 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   order_id           55000 non-null  object        
 1   tanggal_transaksi  55000 non-null  datetime64[ns]
 2   tanggal            55000 non-null  datetime64[ns]
 3   bulan              55000 non-null  object        
 4   bulan_num          55000 non-null  int64         
 5   tahun              55000 non-null  int64         
 6   hari               55000 non-null  object        
 7   jam                55000 non-null  int64         
 8   channel_penjualan  55000 non-null  object        
 9   store_id           55000 non-null  object        
 10  store_name         55000 non-null  object        
 11  tipe_toko          55000 non-null  object        
 12  kota               55000 non-null  object        
 13  provinsi           55000 non-n